# PurpleAir PM2.5 benchmark (2 years, BC + northern WA) on Colab

Trains AirLapseV2 and the baselines on dataset 8: hourly PM2.5 from PurpleAir
sensors in southern BC + northern Washington plus ERA5 weather, Oct 2024 -
Sep 2026 (train Oct 2024 - Dec 2025, val Jan - Mar 2026, test Apr - Sep 2026).
Everything persistent lives in Google Drive under `DRIVE_DIR`, so a
disconnected runtime loses nothing.

**Runtime → Change runtime type → GPU** before running.

The data files are never in the GitHub repo (PurpleAir's terms bar
redistribution). Either upload them to `DRIVE_DIR/data/` (option A) or build
them here (option B).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bcsmoke'   # data/, cache_long/, results/ live here
BRANCH = 'feat/sensor-dataset-4'

import os
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

## 1. Get the code

In [ ]:
%cd /content
!rm -rf PM25GNN && git clone -q -b {BRANCH} https://github.com/Grace-VN/PM25GNN.git
%cd /content/PM25GNN
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## 2. Data

**Option A (recommended):** build on your own machine with
`python pilot/build_bcsmoke_long.py`, then upload `data/BCSmoke2y_nofire.npy`,
`data/BCSmoke2y_monthly.csv` and `data/site_bcsmoke2y.txt` to `DRIVE_DIR/data/`.
Open-Meteo rate-limits per IP and Colab IPs are shared, so building at home is
more predictable.

**Option B:** set `BUILD_HERE = True` and add a Colab secret (key icon in the
left sidebar) `PURPLEAIR_API_KEY`. The cache is on Drive, so if it stops
(disconnect, Open-Meteo daily limit) just run the cell again.

In [ ]:
BUILD_HERE = False

if BUILD_HERE:
    from google.colab import userdata
    os.environ['PURPLEAIR_API_KEY'] = userdata.get('PURPLEAIR_API_KEY')
    !python -u pilot/build_bcsmoke_long.py --cache "{DRIVE_DIR}/cache_long" --out "{DRIVE_DIR}/data"

In [ ]:
# copy the dataset files from Drive into the repo's data/ (where config.yaml points)
!cp -v "{DRIVE_DIR}"/data/BCSmoke2y_* "{DRIVE_DIR}"/data/site_bcsmoke2y.txt data/
import pandas as pd
pd.read_csv('data/BCSmoke2y_monthly.csv', index_col=0)   # PM2.5 level per month

## 3. Benchmark

Results go to `DRIVE_DIR/results/baseline_bench_ds<N>.csv`. Re-running a cell
only retrains the models you list (their old rows are replaced), so after a
disconnect just re-run with the missing models.

Start with 1 repeat to see timings; train.py's output is captured, so the cell
is quiet until each model finishes.

In [ ]:
os.environ['PILOT_RESULTS_DIR'] = f'{DRIVE_DIR}/results'
MODELS = 'AirLapseV2 LSTM GRU PM25_GNN Informer PatchTST'
!python pilot/baseline_bench.py --dataset 8 --models {MODELS} --epochs 50 --repeats 1 --early_stop 10

Raise `--repeats` to 3-5 for the numbers that go in a paper (the first run with 1 repeat tells you the timings).

In [ ]:
import glob
for fp in sorted(glob.glob(f'{DRIVE_DIR}/results/baseline_bench_ds*.csv')):
    df = pd.read_csv(fp)
    df = df[df.status == 'ok']
    print(os.path.basename(fp))
    display(df.groupby('model')[['RMSE', 'RMSE_smoke', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']]
              .mean().round(3).sort_values('RMSE'))